<a href="https://colab.research.google.com/github/lucas-nunn/127050/blob/main/mcnb_stats2_week02_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#IMPORTANT POINT 1
You can't modify this notebook. For this reason, *you need to copy the notebook to your drive. To this end, please go to File->Save a copy in Drive*.

You can make a special folder for the course in your drive. Then, use that copied version.

#IMPORTANT POINT 2
Colab grants us access to GPUs, which makes ANN training much faster. However, after a few hours, access is revoked and we have to wait. For this session, we will not train huge networks, so do not worry if you get a message saying that your GPU hours have been used up. You'll get new GPU hours after a little while.

---

#Week 2: Objectives & network training
Author: Adrien Doerig

\
In this notebook, we will start from where we left off last time, and go into details of how to train and test networks. We will also cover how to use different objectives and optimizers.

---

###Import python packages required for this notebook
***You need to run this cell for the rest of the notebook to work!***


In [ ]:
import torch                              # <- PyTorch
from torch.utils.data import DataLoader
from torch import nn
import torchvision                        # Torch stuff for Computer vision
from torchvision import transforms

import os
import matplotlib.pyplot as plt           # Plotting library
import numpy as np                        # Mathy functions on CPU
from tqdm.notebook import tqdm            # Library to make progress bars

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

##Picking up where we left off

Today, we will focus on objectives, and understanding training/testing loops. We  will do so in the context of our simple MNIST network from last week. The following code cells are copy-pasted from last week, and will create the dataloader, network architecture, and plotting functions that we wrote last week.

>Note: This is copy-pasted code from last week. Feel free to go over it as a recap if you want, but this is not the focus for today.

Get dataset and dataloader

In [ ]:
# Parameters
batch_size = 64

# Transform: convert images to pytorch tensors (cf. description above)
# Note: as we will see, we can add other transforms here for more advanced
# data processing
transform = transforms.Compose([
    transforms.ToTensor()
])

# Download MNIST dataset train set, specify transforms
mnist_train_dataset = torchvision.datasets.MNIST(root='.', # where to download
                                                 train=True, # get train set
                                                 transform=transform, # which transforms to apply
                                                 download=True)

# Create a dataloader, specify batch size, shuffle order
mnist_train_loader = DataLoader(mnist_train_dataset,
                                batch_size=batch_size,
                                shuffle=True)

Create network class

In [ ]:
# nn.Module is the base class for all neural network modules in PyTorch.
# Our SimpleMLP inherits from it.
class SimpleMLP(nn.Module):

    # __init__() method in initializes the layers and attributes of the
    # neural network when an instance is created.
    # Basically, we define all the "building blocks" we need here, and we will
    # combine them in the forward(x) function.
    def __init__(self, n_input_pixels, n_hidden):
        super().__init__()
        self.flatten = nn.Flatten()              # Converts the input image of shape (channels, heigh, width) into a flattened vector of 784 elements.
        self.hidden = nn.Linear(n_input_pixels, n_hidden) # Hidden layer
        self.relu = nn.ReLU()                    # Activation function
        self.output = nn.Linear(n_hidden, 10)    # Output layer
        self.softmax = nn.Softmax(dim=1)         # Softmax activation function

    # forward(x) defines the forward pass of the model, specifying how
    # input x flows through the layers to produce the output.
    def forward(self, x):
        x = self.flatten(x)
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        x = self.softmax(x)
        return x

Get model instance, and put on desired device.

In [ ]:
# Create an instance of the model
mnist_model = SimpleMLP(n_input_pixels=28*28, n_hidden=64)

# Move the model to the desired device
mnist_model = mnist_model.to(device)

Define helper functions for plotting

In [ ]:
def plot_batch_predictions(model, loader, n_imgs):
    # Get a batch of images and labels from the dataloader
    img_batch, label_batch = next(iter(loader))

    # if we only have one color channel, use a grayscal colormap
    # otherwise, use the standard 3-channel one
    if img_batch.shape[1] == 1:
      colormap = 'gray'
    else:
      colormap = None

    # Pass the images through the model to get the predicted labels
    predicted_labels = model(img_batch)

    # Make plot with predicted vs. true labels
    # we'll have max 10 images per row
    n_rows = int(np.ceil(n_imgs/10))
    n_cols = min(10, n_imgs)
    fig, ax = plt.subplots(n_rows, n_cols, figsize=(n_cols,n_rows*1.5))
    if n_rows == 1:
        # Need to make sure ax is 2D array even when there is a single row
        # Otherwise we get wrong indexing in the for loop below
        ax = np.expand_dims(ax, axis=0)

    for i in range(n_imgs):
        row, col = i//10, i%10
        ax[row, col].imshow(img_batch[i].permute(1,2,0), cmap=colormap)
        ax[row, col].set_title(f'pred={torch.argmax(predicted_labels[i])}\ntrue={label_batch[i]}')
        ax[row, col].set_xticks([])
        ax[row, col].set_yticks([])


def plot_dataset_samples(loader, n_imgs):

    # get data from dataloader
    example_data, example_labels = next(iter(loader))

    # if we only have one color channel, use a grayscal colormap
    # otherwise, use the standard 3-channel one
    if example_data.shape[1] == 1:
      colormap = 'gray'
    else:
      colormap = None

    # Make plot with predicted vs. true labels
    # we'll have max 10 images per row
    n_rows = int(np.ceil(n_imgs/10))
    n_cols = min(10, n_imgs)
    fig, ax = plt.subplots(n_rows, n_cols, figsize=(n_cols,n_rows*1.5))
    if n_rows == 1:
        ax = np.expand_dims(ax, axis=0)

    for i in range(n_imgs):
        this_img = example_data[i].permute(1,2,0)
        row, col = i//10, i%10
        ax[row, col].imshow(this_img, cmap=colormap)
        ax[row, col].set_title(f"Label: {example_labels[i]}")
        ax[row, col].set_xticks([])
        ax[row, col].set_yticks([])
    plt.show()

##Objective, training and testing

This week we will dive into code to try different objectives, and to make a more complex training and testing loop.

First, let's define a standard loss function and optimiser to get us started.

**Loss function (or "criterion"):** We will use **cross-entropy**. In brief this loss function says "have a high value for the correct digit class, and a low value for all others".

**Optimiser:** We will use Adam, a widely used optimizer. As we discussed briefly in class, the Adam optimiser is a fancy version of SGB. In a nutshell, it adjusts the learning rate during training in order to get more efficient learning

In [ ]:
mnist_criterion = nn.CrossEntropyLoss()  # Standardly used for classification
mnist_optimizer = torch.optim.Adam(mnist_model.parameters(), lr=0.001)

###Training loop

Let's start by writing a **<font color='red'>training loop</font>**, similar to what we covered in class.

Compared to the code we saw last time, we have two additions:
1. We go over the whole dataset `n_epochs` times.
2. We add a bit of code to remember the loss at each training batch so we can the plot the learning trajectory. We store this in a variable called `history`.


In [ ]:
def train(model, loader, n_epochs, criterion, optimizer, device):
    '''
    model: a pytorch model instance
    loader: a pytorch DataLoader instance
    n_epochs: number of epochs to train for
    criterion: a pytorch loss function instance
    optimizer: a pytorch optimizer instance
    device: 'cuda' or 'cpu'
    '''

    # Set the model to training mode. This helps inform layers such as Dropout
    # and BatchNorm, which are designed to behave differently during training
    # and evaluation. For instance, in training mode, dropout "kills" neurons
    # in each batch, but this is not usually done while testing.
    # In our case, we are not using any of these fancy things, but it is good to
    # get into the habit.
    model.train(True)

    history = []  # we will record the losses over training here.

    # We loop over the whole dataset n_epochs times
    for epoch in range(n_epochs):

        print(f"\nEpoch {epoch + 1}/{n_epochs}")

        # The tqdm python module allows us to have a nice progress bar during
        # training. Don't be confused if it looks complicated.
        # It is just a fancy for loop. We loop over the dataloader and
        # get each item=(batch_imgs, batch_labels) each time.
        for i, item in enumerate(tqdm(loader, desc=f"Epoch {epoch + 1}")):

            optimizer.zero_grad()  # Set all gradients to 0

            inputs, labels = item[0], item[1]  # Pair of input and corresponding label
            inputs = inputs.to(device)         # Move to desired device (GPU/CPU)
            labels = labels.to(device)         # Move to desired device (GPU/CPU)

            outputs = model(inputs)            # Feed the input through the model
            batch_loss = criterion(outputs, labels)  # Calculate the loss

            batch_loss.backward()  # Calculate the gradient for the current loss
            optimizer.step()       # Update weights via backpropagation

            # record loss for this batch as np value. Note that we need all the
            # complicated .cpu().detach().numpy() because the loss is a torch
            # tensor, which includes all the information required to do
            # backprop, use the GPU, etc. We just want normal old numpy.
            history.append(batch_loss.cpu().detach().numpy())

            # Print the current batch loss every few batches
            if i%5==0:
              tqdm.write(f"\rBatch Loss: {batch_loss.item()}", end='')

    return history

>*Quick test:*
1. What does an "epoch" mean?
2. Why do we need `inputs = inputs.to(device)`
3. Point to the line where we pass a batch of inputs and get the model responses.
4. `batch_loss = criterion(outputs, labels)`, `batch_loss.backward()`, and `optimizer.step()` all are involved in training by backpropagation. What is the specific role of each of these steps?


**<font color='red'>Let's train our model!</font>** To do so, we simply use the function containing our training loop defined in the previous code cell.

>**Exercise 1:** Call the training function, specifying that you want to train for 2 epochs.

In [ ]:
# Train!
history = train() # FILL IN THE PARAMETERS

We can now plot the loss curve over training:

In [ ]:
# plot training performance over time
plt.plot(history)
plt.xlabel('batch')
plt.ylabel('loss')

> Exercise: use the helper function we created last time to plot some images and predictions (hint: we re-defined it above in the *Picking up where we left off* section).

In [ ]:
# REPLACE THE FOLLOWING WITH YOUR CODE
raise NotImplementedError

###Testing loop

In machine learning it is common practice to test models not on the data that was used to train them, but instead on held-out test data. Amongst other things, this is done to see how well the model generalizes to unseen data (and does not overfit).

We can download the test set of MNIST, and create a dataloader using the same approach as we did for the training set.

>NOTE: Typically, the training set is split into a training and a smaller validation split. Model selection, and initial analyses are done using the validation set. Then, the test set is only used at the very end. Here, we skip this for simplicity.

---

>**Exercise 2:** Write code to get the test set of MNIST, and make a dataloader for it.
- Note 1: This should be almost identical to the MNIST dataset code from earlier, it only requires minimal changes.
- Note 2: Typically, we don't bother shuffling test sets, so we can get rid of the shuffling in the datalaoder.

**We'll go more in depth into datasets/dataloaders next week**

In [ ]:
transform_list = [transforms.ToTensor()]
transform_test = transforms.Compose(transform_list)

mnist_test_dataset = #TODO: Very similar to how we got the train dataset, but get the test set (look online if needed)

mnist_test_loader = DataLoader(dataset=mnist_test_dataset, batch_size=batch_size,
                               num_workers=2, shuffle=False)

Next, we create a function that evaluates the performance of a model on the test data. There are many ways of assessing model performance. Here, we report the loss, as well as classification accuracy.

>**Exercise 3:** Fill in the blanks in the following code.

In [ ]:
def test(model, loader, criterion, device):
    '''
    model: a pytorch model instance
    loader: a pytorch DataLoader instance
    criterion: a pytorch loss function instance
    device: 'cuda' or 'cpu'
    '''

    # Set the model to evaluation mode. Again, this is important when using
    # more complex ingredients, such as BatchNorm or Dropout, which we are not
    # using here. Still, it is good to get into the habit.
    model.eval()

    # initialize the values we'll report
    test_loss = 0 # this is for the cross-entropy loss
    correct = 0 # we'll get a %correct accuracy
    total = 0 # this is to count the total number of test examples seen.

    # Disable gradient calculation (i.e., we don't keep track of which units
    # impact the loss as we don't want to learn with gradient descent during
    # testing).
    with torch.no_grad():

        # Loop over the testing dataloader, with a nice tqdm progressbar
        for item in tqdm(loader):

            inputs, labels = item[0], item[1]
            inputs = inputs.to(device)
            labels = labels.to(device)

            # pass data through model
            outputs = model(inputs)

            # add loss to total test loss
            test_loss += criterion(outputs, labels).item() # note: a += b is the same as a = a+b

            # to compute the accuracy, we need to get the output with the
            # highest value (this is the network's prediction)
            _, predicted = torch.max(outputs.data, 1)
            # this is a logical operation to get the number of items
            # where the prediction matches the label
            correct += (predicted == labels).sum().item()

            # keep track of how many test samples we've seen
            total += labels.size(0)

    test_loss = # TODO: report average loss per batch
    accuracy = # TODO: report percent correct
    print(f"Test Loss: {test_loss:.4f}, Accuracy: {accuracy:.2f}%")

    return test_loss, accuracy

> **Exercise 4:** Use this function to check the test performance of your network.

In [ ]:
# REPLACE THE FOLLOWING WITH YOUR CODE
raise NotImplementedError

Now that we have a nice testing function, we can improve our training function to print the test loss after each epoch. This is useful so that we get a feeling for how well our model does on held out data during training. For example, this could be used to stop training when the test set starts going up (indicative of overfitting).

Also it's just a nice and simple way to train and test a model all in one go.

>**Exercise 5:** Fill in the blanks in the following code.
- The idea is to re-write our training loop function such that it runs the test loop function after eahc epoch.
- Therefore, your job is mostly to copy-paste the training loop and insert the testing loop at the right place.

In [ ]:
# improve our training function to print the test loss after each epoch.
def train_and_test(model, dataloader_train, dataloader_test, n_epochs, criterion, optimizer, device):
    '''
    model: a pytorch model instance
    dataloader_train: a pytorch DataLoader instance for training
    dataloader_test: a pytorch DataLoader instance for testing
    n_epochs: number of epochs to train for
    criterion: a pytorch loss function instance
    optimizer: a pytorch optimizer instance
    device: 'cuda' or 'cpu'
    '''

    history = []

    # We loop over the whole training dataset n_epochs times
    for epoch in range(n_epochs):

        print(f"Epoch {epoch + 1}/{n_epochs}")

        # Set the model to training mode.
        model.train(True)

        # Do training loop.
        # THIS IS ESSENTIALLY THE SAME AS OUR PREVIOUS TRAINING FUNCTION
        for i, item in enumerate(tqdm(dataloader_train, desc=f"Epoch {epoch + 1}")):
            # TODO: Get training loop contents from out previous train() function

        # After each epoch, evaluate the model on the test set
        test_loss, accuracy = #TODO: call the test() function we just created

    return history

> **Exercise 6:** Let's try it out! To this end, let's reinitialize our network so we re-start training from scratch, and then call this function. Here is a step by step guide:
1. Our `mnist_model` network instance has already been trained, so we need to reinitialize it.
2. Do so by re-creating the model instance and re-putting it onto the right device.
3. Re-create the loss and Adam optimizer, and pass the new model's parameters to the optimizer.
4. Call the `train_and_test()` function to train and test your new model instance.
5. Plot the loss curve, using the returned `history` (similarly to what we did previously in this notebook).

In [ ]:
# Re-create model instance and move to desired device
mnist_model = #TODO create new model instance
mnist_model = mnist_model.to(device) # puts new instance on GPU if available

# Loss and optimizer
mnist_criterion = #TODO make cross entropy loss as we did previously
mnist_optimizer = #TODO make Adam optimiser with learning rate = 0.001 we did previously

history = #TODO: call the train_and_test() function we just defined

# plot training performance over time
plt.plot(history)
plt.xlabel('batch')
plt.ylabel('loss')

##Exercise 7: Explore optimisers and learning rates

Repeat the steps of exercise 6, while varying the optimiser type and the learning rater. In particular:

1. Try standard Stochastic Gradient Descent (see [pytorch documentation](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html)) instead of Adam.
2. For both optimisers, try various learning rates (i.e., the `lr` parameter of the pytorch optimiser). Can you reproduce the theory covered in class, where we discussed that:
  - an excessively low lr leads to slower training.
  - an excessively high lr leads to bad performance.

In [ ]:
# Re-create model instance and move to desired device
mnist_model = #TODO create new model instance
mnist_model = mnist_model.to(device)

# Loss and optimizer
mnist_criterion = nn.CrossEntropyLoss()  # Standardly used for classification
mnist_optimizer = #TODO make SGD optimiser, try various learning rates

history = #TODO: call the train_and_test() function we just defined

# plot training performance over time
plt.plot(history)
plt.xlabel('batch')
plt.ylabel('loss')

##Exercise 8: Different objectives

The cross-entropy objective we have been using is the most widely used for classification. In short it says "network output should be high for the tru class, and low for others.

Another possibility would be the Negative Log Likelihood Loss (NLLL), which implements the maximum likelihood estimation technique which we discussed last semester in the context of logistic regression (there are differences since this is no longer logistic regression, but it's the same underlying idea).

The PyTorch loss for NLLL is called nn.NLLLoss(). Modify your code from the previous exercise to use the negative Log-Likelihood loss.


In [ ]:
# Re-create model instance and move to desired device
mnist_model = #TODO create new model instance
mnist_model = mnist_model.to(device)

# Loss and optimizer
mnist_criterion = #TODO make NLLL optimiser
mnist_optimizer = torch.optim.Adam(mnist_model.parameters(), lr=0.01)

history = #TODO: call the train_and_test() function we just defined

# plot training performance over time
plt.plot(history)
plt.xlabel('batch')
plt.ylabel('loss')